<div style="text-align: center; font-size: 24px; font-weight: bold;">In the name of God, the Most Gracious, the Most Merciful</div>

Full Name: Mohammadmahdi Bababeyk

Student ID: 4041419005

# Building a Depthwise–Separable ResNet-18 for CIFAR-10

**Objective:**

- You will implement a lightweight ResNet-18 where all 3×3 convolutions are replaced by depthwise + pointwise convolutions (MobileNet-style).
- You will then train and evaluate the model on the CIFAR-10 dataset.

### Practice 1: Implementing Depthwise + Pointwise Convolution

In this part, you will rewrite the basic 3×3 convolution from ResNet into a depthwise–separable convolution. This component is the heart of the entire assignment.

**1. Depthwise Convolution (3×3)**

You must implement a 2D depthwise convolution layer with:

- Kernel size: 3×3

- Padding: 1 (to keep spatial size)

- Stride: variable (depends on block configuration)

- groups = in_channels → this is the key point

- No bias

**Important details:**

- A standard 3×3 convolution mixes spatial information and channel information simultaneously.

- A depthwise convolution only applies a 3×3 filter per channel independently.

- So if the input has C channels, and you perform a depthwise 3×3:

    - You get C “filtered” channels

    - You learn C filters, one per channel

- This drastically reduces computation:

    - Standard conv: `3 × 3 × Cin × Cout`

    - Depthwise: `3 × 3 × Cin`

You should pay attention to:

- The number of groups MUST equal the number of input channels.

- Padding must be exactly 1 so that 32×32 CIFAR images remain aligned in deeper layers.

- Stride must match the downsampling requirements (1 or 2).

**2. Pointwise Convolution (1×1)**

After the depthwise conv, implement a 1×1 convolution with:

- No bias

- `in_channels → out_channels`

- The purpose is to mix information across channels, which depthwise conv alone cannot do.

**Nano-details:**

- A 1×1 convolution is mathematically equivalent to a per-pixel fully connected layer applied across channels.

- This provides the cross-channel interaction missing in the depthwise stage.

- This pair (DWConv + PWConv) is how MobileNet-style separable convolutions achieve:

    - High accuracy

    - Very low computation

**3. Batch Normalization + ReLU**

After the 1×1 convolution:

- Apply BatchNorm2d(out_channels)

- Then apply a ReLU activation (non-inplace to avoid graph issues)

**Nano-details:**

- BatchNorm stabilizes training by normalizing each channel independently.

- Must be applied after pointwise conv (not before).

- ReLU must come after BatchNorm (ResNet standard ordering).

**4. Forward Pass of Depthwise–Separable Block**

Your block should perform:

    Input → Depthwise 3×3 → Pointwise 1×1 → BatchNorm → ReLU

Make sure the forward pass calls each component in the correct order.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import torch  # Import PyTorch for tensor operations
import torch.nn as nn  # Import PyTorch's neural network module for building layers
import torchvision  # Import torchvision for datasets and transforms
import torchvision.transforms as transforms  # Import transforms module for data augmentation and normalization
from torch.utils.data import DataLoader  # Import DataLoader for creating iterable data loaders
import torch.optim as optim  # Import optim module for optimizers
from torch.optim.lr_scheduler import MultiStepLR  # Import MultiStepLR for learning rate scheduling

class SeparableConv2d(nn.Module):
    """
    Depthwise-Separable Convolution module.

    This class replaces a standard 3x3 convolution with a depthwise 3x3 convolution
    followed by a pointwise 1x1 convolution, then Batch Normalization and ReLU activation.
    It is designed to mimic MobileNet-style separable convolutions for efficiency.

    Args:
        in_channels (int): Number of input channels.
        out_channels (int): Number of output channels.
        stride (int, optional): Stride for the depthwise convolution. Defaults to 1.
    """

    def __init__(self, in_channels, out_channels, stride=1):
        # Call the parent class (nn.Module) initializer
        super(SeparableConv2d, self).__init__()
        # Define the depthwise convolution: 3x3 kernel, padding 1 to maintain spatial size,
        # stride as specified, groups equal to in_channels for per-channel filtering, no bias
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, stride=stride, groups=in_channels, bias=False)
        # Define the pointwise convolution: 1x1 kernel to mix channels, no bias
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        # Define batch normalization for the output channels to stabilize training
        self.bn = nn.BatchNorm2d(out_channels)
        # Define ReLU activation (non-inplace by default to avoid issues with computation graph)
        self.relu = nn.ReLU()

    def forward(self, x):
        """
        Forward pass of the SeparableConv2d module.

        Applies depthwise convolution, pointwise convolution, batch normalization,
        and ReLU activation in sequence.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_channels, height, width).

        Returns:
            torch.Tensor: Output tensor of shape (batch_size, out_channels, height', width'),
                          where height' and width' depend on the stride.
        """
        # Apply the depthwise 3x3 convolution to the input
        x = self.depthwise(x)
        # Apply the pointwise 1x1 convolution to mix channel information
        x = self.pointwise(x)
        # Apply batch normalization to normalize the output channels
        x = self.bn(x)
        # Apply ReLU activation to introduce non-linearity
        x = self.relu(x)
        # Return the resulting tensor
        return x

### Practice 2: Implementing a Depthwise–Separable ResNet Basic Block


You must re-implement the BasicBlock of ResNet-18 using your DWConv3x3 block:

- First convolution: `DWConv3x3(in_channels → out_channels)`

- Second convolution: `DWConv3x3(out_channels → out_channels)`

- Optional 1×1 downsample layer when:

    - stride = 2, or

    - input channels ≠ output channels

Your tasks:

- Build the two separable convolution layers.

- Handle the identity shortcut correctly.

- Add the output and the identity.

- Apply ReLU at the end.

In [3]:
class BasicBlock(nn.Module):
    """
    Depthwise-Separable ResNet Basic Block.

    This class implements a basic residual block for ResNet-18 where the standard
    3x3 convolutions are replaced with depthwise-separable convolutions.
    It uses two SeparableConv2d layers: the first with possible downsampling stride,
    the second with stride 1. To match the ResNet structure, the ReLU of the second
    SeparableConv2d is skipped before adding the shortcut, and a final ReLU is applied
    after the addition.

    Args:
        in_channels (int): Number of input channels.
        out_channels (int): Number of output channels.
        stride (int, optional): Stride for the first convolution. Defaults to 1.
    """

    def __init__(self, in_channels, out_channels, stride=1):
        # Call the parent class (nn.Module) initializer
        super(BasicBlock, self).__init__()
        # First separable convolution with possible stride for downsampling
        self.conv1 = SeparableConv2d(in_channels, out_channels, stride=stride)
        # Second separable convolution with stride 1
        self.conv2 = SeparableConv2d(out_channels, out_channels, stride=1)
        # Initialize downsample as None
        self.downsample = None
        # Check if downsample is needed (for channel mismatch or spatial downsampling)
        if stride != 1 or in_channels != out_channels:
            # Define downsample as 1x1 conv + batch norm if needed
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),  # 1x1 conv to match channels and stride
                nn.BatchNorm2d(out_channels)  # Batch norm for the downsampled shortcut
            )
        # Define the final ReLU activation after addition
        self.relu = nn.ReLU()

    def forward(self, x):
        """
        Forward pass of the BasicBlock.

        Applies the two separable convolutions, handles the shortcut connection,
        adds the shortcut to the main path, and applies a final ReLU.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_channels, height, width).

        Returns:
            torch.Tensor: Output tensor of shape (batch_size, out_channels, height', width').
        """
        # Store the identity (shortcut) as the input
        identity = x
        # Apply downsample to identity if needed
        if self.downsample is not None:
            identity = self.downsample(identity)
        # Apply the first separable conv (includes DW, PW, BN, ReLU)
        x = self.conv1(x)
        # Apply the depthwise conv of the second separable
        x = self.conv2.depthwise(x)
        # Apply the pointwise conv of the second separable
        x = self.conv2.pointwise(x)
        # Apply the batch norm of the second separable
        x = self.conv2.bn(x)
        # Add the shortcut to the main path
        x += identity
        # Apply the final ReLU activation
        x = self.relu(x)
        # Return the resulting tensor
        return x

### Practice 3: Building the Full Depthwise–Separable ResNet-18

Construct the full model by stacking the BasicBlockDS you implemented.

**Architecture Layout (for CIFAR-10):**

- Conv1: 3×3 conv producing 64 channels

- Layer1: 2 blocks, stride 1

- Layer2: 2 blocks, stride 2

- Layer3: 2 blocks, stride 2

- Layer4: 2 blocks, stride 2

Then:

- A final 1×1 convolution to map from 512 → 10 classes

- Adaptive average pooling to 1×1

- Flatten for classification

You must:

- Implement `_make_layer()` to create `N` blocks per stage

- Properly update `self.in_channels` for each stage

- Initialize all conv and BN layers according to Kaiming rules

In [4]:
class DSResNet(nn.Module):
    """
    Depthwise-Separable ResNet-18 model for CIFAR-10.

    This class implements a modified ResNet-18 where the 3x3 convolutions in the
    residual blocks are replaced with depthwise-separable convolutions. The model
    is adapted for CIFAR-10 with a smaller initial convolution and no max pooling.
    It uses a 1x1 convolution as the final classifier instead of a linear layer.

    The architecture includes:
    - Initial 3x3 convolution to 64 channels.
    - Four layers of BasicBlocks with increasing channels (64, 128, 256, 512).
    - Adaptive average pooling to 1x1.
    - 1x1 convolution to 10 classes.
    - Flatten to produce logits.
    """

    def __init__(self, num_classes=10):
        # Call the parent class (nn.Module) initializer
        super(DSResNet, self).__init__()
        # Set initial in_channels after the first convolution
        self.in_channels = 64
        # Define the initial 3x3 convolution: input 3 channels, output 64, kernel 3, stride 1, padding 1, no bias
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        # Define batch normalization after the initial convolution
        self.bn1 = nn.BatchNorm2d(64)
        # Define ReLU activation after the initial BN
        self.relu = nn.ReLU()
        # Create layer1: 2 blocks with 64 channels, stride 1
        self.layer1 = self._make_layer(64, 2, stride=1)
        # Create layer2: 2 blocks with 128 channels, stride 2 for downsampling
        self.layer2 = self._make_layer(128, 2, stride=2)
        # Create layer3: 2 blocks with 256 channels, stride 2 for downsampling
        self.layer3 = self._make_layer(256, 2, stride=2)
        # Create layer4: 2 blocks with 512 channels, stride 2 for downsampling
        self.layer4 = self._make_layer(512, 2, stride=2)
        # Define adaptive average pooling to reduce spatial dimensions to 1x1
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        # Define final 1x1 convolution to map 512 channels to num_classes (10 for CIFAR-10), no bias by default
        self.fc = nn.Conv2d(512, num_classes, kernel_size=1)
        # Initialize weights according to Kaiming rules
        self._init_weights()

    def _make_layer(self, out_channels, num_blocks, stride):
        """
        Helper function to create a layer consisting of multiple BasicBlocks.

        Args:
            out_channels (int): Number of output channels for the blocks in this layer.
            num_blocks (int): Number of BasicBlocks in this layer.
            stride (int): Stride for the first block (for downsampling if >1).

        Returns:
            nn.Sequential: A sequential container of the BasicBlocks.
        """
        # Initialize an empty list to hold the blocks
        layers = []
        # Append the first block with the given stride (may downsample)
        layers.append(BasicBlock(self.in_channels, out_channels, stride))
        # Update in_channels to out_channels after the first block
        self.in_channels = out_channels
        # Append the remaining blocks with stride 1
        for _ in range(1, num_blocks):
            layers.append(BasicBlock(self.in_channels, out_channels, 1))
        # Return the layers as a sequential module
        return nn.Sequential(*layers)

    def _init_weights(self):
        """
        Initialize weights for all Conv2d and BatchNorm2d layers using Kaiming initialization.
        """
        # Iterate over all modules in the network
        for m in self.modules():
            # Check if the module is a Conv2d
            if isinstance(m, nn.Conv2d):
                # Apply Kaiming normal initialization to the weights
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            # Check if the module is a BatchNorm2d
            elif isinstance(m, nn.BatchNorm2d):
                # Initialize BN weights to 1
                nn.init.constant_(m.weight, 1)
                # Initialize BN biases to 0
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        """
        Forward pass of the DSResNet model.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, 3, height, width).

        Returns:
            torch.Tensor: Output tensor of shape (batch_size, num_classes).
        """
        # Apply the initial 3x3 convolution
        x = self.conv1(x)
        # Apply batch normalization after initial conv
        x = self.bn1(x)
        # Apply ReLU activation
        x = self.relu(x)
        # Pass through layer1
        x = self.layer1(x)
        # Pass through layer2
        x = self.layer2(x)
        # Pass through layer3
        x = self.layer3(x)
        # Pass through layer4
        x = self.layer4(x)
        # Apply adaptive average pooling to 1x1
        x = self.avgpool(x)
        # Apply the final 1x1 convolution for classification
        x = self.fc(x)
        # Flatten the output starting from dimension 1 (channels) to get (batch_size, num_classes)
        x = torch.flatten(x, 1)
        # Return the output logits
        return x

### Practice 4: Preparing the CIFAR-10 Dataset

You must:

1. Load CIFAR-10 using torchvision.datasets.CIFAR10

2. Apply standard augmentations for training:

    - Random crop
    
    - Random horizontal flip

3. Normalize using the CIFAR-10 mean and std

4. Build DataLoaders for train and test sets

Make sure to use:

- Batch size = 128

- Shuffle = True for training

- 2 worker threads

In [5]:
def get_cifar10_loaders(batch_size=128, num_workers=2):
    """
    Prepare and return DataLoaders for CIFAR-10 train and test sets.

    This function loads the CIFAR-10 dataset, applies standard augmentations
    and normalization for training and testing, and creates DataLoaders.

    Args:
        batch_size (int, optional): Batch size for the loaders. Defaults to 128.
        num_workers (int, optional): Number of worker threads for data loading. Defaults to 2.

    Returns:
        tuple: (train_loader, test_loader)
    """
    # Define the training transforms: compose a sequence of augmentations
    train_transform = transforms.Compose([
        # Randomly crop the image to 32x32 with padding of 4 pixels on each side
        transforms.RandomCrop(32, padding=4),
        # Randomly flip the image horizontally with 50% probability
        transforms.RandomHorizontalFlip(),
        # Convert the image to a PyTorch tensor
        transforms.ToTensor(),
        # Normalize the tensor with CIFAR-10 mean and std for each channel
        transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.2010])
    ])
    # Define the test transforms: no augmentations, only normalization
    test_transform = transforms.Compose([
        # Convert the image to a PyTorch tensor
        transforms.ToTensor(),
        # Normalize the tensor with CIFAR-10 mean and std for each channel
        transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2023, 0.1994, 0.2010])
    ])
    # Load the training dataset from CIFAR-10, download if not present
    train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
    # Load the test dataset from CIFAR-10, download if not present
    test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
    # Create the training DataLoader with shuffling enabled
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    # Create the test DataLoader without shuffling
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    # Return the train and test loaders
    return train_loader, test_loader

### Practice 5: Training One Epoch

Implement a function that:

1. Sets model to training mode

2. Iterates through all batches:

    - Forward pass
    
    - Compute loss
    
    - Backward pass
    
    - Optimizer step

3. Tracks:

    - Total loss
    
    - Classification accuracy

Return:

- epoch loss

- epoch accuracy

In [6]:
def train_one_epoch(model, train_loader, optimizer, criterion, device='cpu'):
    """
    Train the model for one epoch on the given dataset.

    This function sets the model to training mode, iterates through the training
    data loader, performs forward and backward passes, updates the model parameters,
    and tracks the average loss and accuracy for the epoch.

    Args:
        model (nn.Module): The neural network model to train.
        train_loader (DataLoader): DataLoader for the training dataset.
        optimizer (torch.optim.Optimizer): Optimizer for updating model parameters.
        criterion (nn.Module): Loss function (e.g., nn.CrossEntropyLoss).
        device (str, optional): Device to run the training on ('cpu' or 'cuda'). Defaults to 'cpu'.

    Returns:
        tuple: (epoch_loss, epoch_acc) where epoch_loss is the average loss,
               and epoch_acc is the accuracy for the epoch.
    """
    # Set the model to training mode (enables dropout, batch norm updates, etc.)
    model.train()
    # Initialize total loss accumulator
    total_loss = 0.0
    # Initialize counter for correct predictions
    correct = 0
    # Initialize counter for total samples processed
    total = 0
    # Iterate over each batch in the training loader
    for data, target in train_loader:
        # Move input data to the specified device
        data = data.to(device)
        # Move target labels to the specified device
        target = target.to(device)
        # Zero the gradients in the optimizer before forward pass
        optimizer.zero_grad()
        # Perform forward pass through the model
        output = model(data)
        # Compute the loss between output and target
        loss = criterion(output, target)
        # Perform backward pass to compute gradients
        loss.backward()
        # Update model parameters using the optimizer
        optimizer.step()
        # Accumulate the loss value (detach from graph with .item())
        total_loss += loss.item()
        # Get the predicted class indices (max logit per sample)
        _, predicted = output.max(1)
        # Update total samples count
        total += target.size(0)
        # Count correct predictions and accumulate
        correct += predicted.eq(target).sum().item()
    # Calculate average loss over all batches
    epoch_loss = total_loss / len(train_loader)
    # Calculate accuracy as correct over total samples
    epoch_acc = correct / total
    # Return the epoch loss and accuracy
    return epoch_loss, epoch_acc

### Practice 6: Evaluation Loop and Final Training Script

You must:

**Evaluation function**

- Disable gradients with torch.no_grad()

- Loop over the test dataset

- Compute:

    - Loss

    - Accuracy

- Model stays in .eval() mode

**Main training loop**

Run for 20 epochs:

- Train

- Evaluate

- Step the LR scheduler

- Save the model when improvement occurs

At the end:

- Print best achieved accuracy

In [9]:
def evaluate(model, test_loader, criterion, device='cpu'):
    """
    Evaluate the model on the test dataset.

    This function sets the model to evaluation mode, disables gradients,
    iterates through the test data loader, and computes the average loss
    and accuracy without updating the model parameters.

    Args:
        model (nn.Module): The neural network model to evaluate.
        test_loader (DataLoader): DataLoader for the test dataset.
        criterion (nn.Module): Loss function (e.g., nn.CrossEntropyLoss).
        device (str, optional): Device to run the evaluation on ('cpu' or 'cuda'). Defaults to 'cpu'.

    Returns:
        tuple: (eval_loss, eval_acc) where eval_loss is the average loss,
               and eval_acc is the accuracy on the test set.
    """
    # Set the model to evaluation mode (disables dropout, freezes batch norm stats)
    model.eval()
    # Initialize total loss accumulator
    total_loss = 0.0
    # Initialize counter for correct predictions
    correct = 0
    # Initialize counter for total samples processed
    total = 0
    # Disable gradient computation for efficiency
    with torch.no_grad():
        # Iterate over each batch in the test loader
        for data, target in test_loader:
            # Move input data to the specified device
            data = data.to(device)
            # Move target labels to the specified device
            target = target.to(device)
            # Perform forward pass through the model
            output = model(data)
            # Compute the loss between output and target
            loss = criterion(output, target)
            # Accumulate the loss value (detach from graph with .item())
            total_loss += loss.item()
            # Get the predicted class indices (max logit per sample)
            _, predicted = output.max(1)
            # Update total samples count
            total += target.size(0)
            # Count correct predictions and accumulate
            correct += predicted.eq(target).sum().item()
    # Calculate average loss over all batches
    eval_loss = total_loss / len(test_loader)
    # Calculate accuracy as correct over total samples
    eval_acc = correct / total
    # Return the evaluation loss and accuracy
    return eval_loss, eval_acc

# Main training script
if __name__ == "__main__":
    # Determine the device: use CUDA if available, else CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Get the train and test data loaders
    train_loader, test_loader = get_cifar10_loaders()
    # Initialize the model and move to the device
    model = DSResNet().to(device)
    # Define the optimizer: SGD with learning rate 0.1, momentum 0.9, weight decay 5e-4
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    # Define the loss criterion: Cross Entropy Loss
    criterion = nn.CrossEntropyLoss()
    # Define the learning rate scheduler: MultiStepLR with milestones at epochs 10 and 15, gamma 0.1
    scheduler = MultiStepLR(optimizer, milestones=[10, 15], gamma=0.1)
    # Initialize best accuracy tracker
    best_acc = 0.0
    # Loop over 20 epochs
    for epoch in range(20):
        # Train for one epoch and get train loss and accuracy
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        # Evaluate on test set and get eval loss and accuracy
        eval_loss, eval_acc = evaluate(model, test_loader, criterion, device)
        # Step the learning rate scheduler
        scheduler.step()
        # Check if current accuracy is better than best
        if eval_acc > best_acc:
            # Update best accuracy
            best_acc = eval_acc
            # Save the model state dict
            torch.save(model.state_dict(), 'best_model.pth')
        # Print epoch summary
        print(f"Epoch {epoch+1}: Train Loss {train_loss:.4f}, Train Acc {train_acc:.4f}, Eval Loss {eval_loss:.4f}, Eval Acc {eval_acc:.4f}")
    # Print the best achieved accuracy at the end
    print(f"Best achieved accuracy: {best_acc:.4f}")

Epoch 1: Train Loss 2.8168, Train Acc 0.2218, Eval Loss 1.8936, Eval Acc 0.2937
Epoch 2: Train Loss 1.7695, Train Acc 0.3362, Eval Loss 1.6227, Eval Acc 0.3945
Epoch 3: Train Loss 1.5342, Train Acc 0.4347, Eval Loss 1.4414, Eval Acc 0.4765
Epoch 4: Train Loss 1.3347, Train Acc 0.5150, Eval Loss 1.4478, Eval Acc 0.4999
Epoch 5: Train Loss 1.1736, Train Acc 0.5806, Eval Loss 1.1416, Eval Acc 0.5970
Epoch 6: Train Loss 1.0382, Train Acc 0.6308, Eval Loss 1.2206, Eval Acc 0.5848
Epoch 7: Train Loss 0.9551, Train Acc 0.6639, Eval Loss 1.1254, Eval Acc 0.6297
Epoch 8: Train Loss 0.8778, Train Acc 0.6905, Eval Loss 1.0589, Eval Acc 0.6391
Epoch 9: Train Loss 0.8191, Train Acc 0.7122, Eval Loss 1.0612, Eval Acc 0.6277
Epoch 10: Train Loss 0.7661, Train Acc 0.7325, Eval Loss 0.7990, Eval Acc 0.7211
Epoch 11: Train Loss 0.5404, Train Acc 0.8132, Eval Loss 0.5323, Eval Acc 0.8167
Epoch 12: Train Loss 0.4800, Train Acc 0.8333, Eval Loss 0.5076, Eval Acc 0.8234
Epoch 13: Train Loss 0.4486, Train Ac

---
**Note:** This notebook is part of a Deep Learning assignment designed and prepared by [Mahdi Golizadeh](mailto:mahdi.golizadeh@gmail.com).